# 📊 Model Evaluation — Energy Grid Outage Prediction

Confusion matrix, feature importance, and prediction analysis.

**Reads from:** `gold_ml_features`, saved model

**Writes to:** `gold_ml_feature_importance`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.ml.feature import VectorAssembler, StringIndexer
from synapse.ml.lightgbm import LightGBMClassificationModel

spark = SparkSession.builder.getOrCreate()
model = LightGBMClassificationModel.load('Files/models/outage_prediction_lgbm')
print('Model loaded')

In [ ]:
features_df = spark.read.table('gold_ml_features')

numeric_features = [
    'avg_voltage', 'std_voltage', 'min_voltage', 'max_voltage', 'voltage_range', 'voltage_deviation',
    'avg_frequency', 'std_frequency', 'freq_deviation',
    'avg_power_factor', 'min_power_factor',
    'avg_load', 'max_load',
    'avg_temp', 'max_temp',
    'reading_count', 'day_of_week', 'month',
]

indexer_region = StringIndexer(inputCol='region', outputCol='region_idx', handleInvalid='keep')
indexed_df = indexer_region.fit(features_df).transform(features_df)
all_features = numeric_features + ['region_idx']
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='skip')
model_df = assembler.transform(indexed_df)
_, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
predictions = model.transform(test_df)
print(f'Test predictions: {predictions.count():,} rows')

In [ ]:
print('=== Confusion Matrix ===')
predictions.groupBy('had_outage', 'prediction').count().orderBy('had_outage', 'prediction').show()

tp = predictions.filter((col('had_outage') == 1) & (col('prediction') == 1)).count()
fp = predictions.filter((col('had_outage') == 0) & (col('prediction') == 1)).count()
fn = predictions.filter((col('had_outage') == 1) & (col('prediction') == 0)).count()
tn = predictions.filter((col('had_outage') == 0) & (col('prediction') == 0)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')

In [ ]:
import pandas as pd

try:
    importances = model.getFeatureImportances(importance_type='gain')
    fi_df = pd.DataFrame({'feature': all_features, 'importance': importances[:len(all_features)]})
    fi_df = fi_df.sort_values('importance', ascending=False)
    print('=== Top 10 Features ===')
    for _, row in fi_df.head(10).iterrows():
        print(f"  {row['feature']:30s} {row['importance']:.4f}")
    fi_spark = spark.createDataFrame(fi_df).withColumn('model_timestamp', current_timestamp())
    fi_spark.write.mode('overwrite').format('delta').saveAsTable('gold_ml_feature_importance')
    print('Feature importance saved')
except Exception as e:
    print(f'Could not extract feature importance: {e}')
    fi_data = [(f, 0.0) for f in all_features]
    fi_spark = spark.createDataFrame(fi_data, ['feature', 'importance']).withColumn('model_timestamp', current_timestamp())
    fi_spark.write.mode('overwrite').format('delta').saveAsTable('gold_ml_feature_importance')

In [ ]:
spark.sql('OPTIMIZE gold_ml_feature_importance')
print('Evaluation complete')